In [1]:
# als-baseline.ipnyb
# Скачивание дополнительных пакетов
%pip install scipy==1.11.4
%pip install -q transformers accelerate bitsandbytes torch sentence-transformers faiss-cpu pandas numpy seaborn
%pip install implicit
# Импорт библиотек
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
import numpy as np
from collections import Counter
from transformers import AutoTokenizer, AutoModelForCausalLM, BitsAndBytesConfig
import torch
import json
import re
from scipy.sparse import csr_matrix
from sklearn.decomposition import TruncatedSVD
from sklearn.preprocessing import StandardScaler
from implicit.als import AlternatingLeastSquares
import joblib
import pickle
import os
from implicit.evaluation import train_test_split, precision_at_k, ndcg_at_k
import warnings
warnings.filterwarnings('ignore')


# Датасет можно скачать дополнительно тут:
# https://www.kaggle.com/datasets/zygmunt/goodbooks-10k/data

# Пути к файлам
DATA_PATH = "../data"
# DATA_PATH = "/content"
ratings = pd.read_csv(f"{DATA_PATH}/ratings.csv")
books = pd.read_csv(f"{DATA_PATH}/books.csv")
to_read = pd.read_csv(f"{DATA_PATH}/to_read.csv")
tags = pd.read_csv(f"{DATA_PATH}/tags.csv")
book_tags = pd.read_csv(f"{DATA_PATH}/book_tags.csv")

print("Данные загружены\n")

ARTIFACTS_PATH = "../artifacts/als_models"
print(f"Путь для сохранения артефактов: {ARTIFACTS_PATH}")
os.makedirs(ARTIFACTS_PATH, exist_ok=True)

SRC_PATH = "../src/models"
print(f"Путь для сохранения лучшей модели: {SRC_PATH}")
os.makedirs(SRC_PATH, exist_ok=True)

Note: you may need to restart the kernel to use updated packages.



[notice] A new release of pip is available: 25.3 -> 26.0.1
[notice] To update, run: python.exe -m pip install --upgrade pip


Note: you may need to restart the kernel to use updated packages.



[notice] A new release of pip is available: 25.3 -> 26.0.1
[notice] To update, run: python.exe -m pip install --upgrade pip


Note: you may need to restart the kernel to use updated packages.



[notice] A new release of pip is available: 25.3 -> 26.0.1
[notice] To update, run: python.exe -m pip install --upgrade pip
c:\Users\andre\AppData\Local\Programs\Python\Python311\Lib\site-packages\tqdm\auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


Данные загружены

Путь для сохранения артефактов: ../artifacts/als_models
Путь для сохранения лучшей модели: ../src/models


In [2]:
# Маппинг ID в непрерывный диапазон 0..N-1 (требуется implicit)
user_ids = ratings['user_id'].unique()
book_ids = ratings['book_id'].unique()
user_id_map = {uid: i for i, uid in enumerate(user_ids)}
book_id_map = {bid: i for i, bid in enumerate(book_ids)}
user_id_inv = {i: uid for uid, i in user_id_map.items()}
book_id_inv = {i: bid for bid, i in book_id_map.items()}

ratings['user_idx'] = ratings['user_id'].map(user_id_map)
ratings['item_idx'] = ratings['book_id'].map(book_id_map)

# Преобразуем оценки в бинарные взаимодействия (1 = пользователь оценил книгу)
csr_data = csr_matrix(
    (np.ones(len(ratings)), (ratings['user_idx'].values, ratings['item_idx'].values)),
    shape=(len(user_ids), len(book_ids))
)

# Разделение на train/test
print("Разделение данных (80/20)...")
train, test = train_test_split(csr_data, train_percentage=0.8)

# --- Расширенная сетка гиперпараметров ---
param_grid = [
    {'factors': 32,  'regularization': 0.01, 'iterations': 30, 'alpha': 1.0}
]

print(f"Всего конфигураций для обучения: {len(param_grid)}")    

# --- Цикл обучения и оценки ---
results = []
K = 10  # Для метрик @K

for i, params in enumerate(param_grid):
    print(f"\n[Model {i+1}/{len(param_grid)}] Обучение с параметрами: {params}")
    model = AlternatingLeastSquares(**params)
    model.fit(train, show_progress=True)

    # Расчет метрик
    p_k   = precision_at_k(model, train, test, K=K, show_progress=False)
    ndcg_k = ndcg_at_k(model, train, test, K=K, show_progress=False)

    metrics = {
        'model_id': i + 1,
        'factors': params['factors'],
        'regularization': params['regularization'],
        'iterations': params['iterations'],
        'alpha': params['alpha'],
        'Precision@K': p_k,
        'NDCG@K': ndcg_k,
    }
    results.append(metrics)

    # Сохранение каждой модели в отдельную папку
    model_dir = os.path.join(ARTIFACTS_PATH, f"model_{i+1}")
    os.makedirs(model_dir, exist_ok=True)
    np.save(os.path.join(model_dir, "user_factors.npy"), model.user_factors)
    np.save(os.path.join(model_dir, "item_factors.npy"), model.item_factors)
    with open(os.path.join(model_dir, "params.pkl"), 'wb') as f:
        pickle.dump(params, f)

# --- Вывод таблицы метрик ---
results_df = pd.DataFrame(results)
print("\n" + "="*90)
print("СВОДНАЯ ТАБЛИЦА МЕТРИК")
print("="*90)
print(results_df.to_string(index=False))
print("="*90)

# --- Выбор лучшей модели и сохранение ---
# Основной критерий: NDCG@10 
best_idx = results_df['NDCG@K'].idxmax()
best_row = results_df.loc[best_idx]
model_id_int = int(best_row['model_id'])
best_params = param_grid[model_id_int - 1]

print(f"\nЛУЧШАЯ МОДЕЛЬ (по NDCG@{K}): Model {best_row['model_id']}")
print(best_row.drop('model_id').to_string())

print("\nПереобучение лучшей модели на full train и сохранение артефактов...")
best_model = AlternatingLeastSquares(**best_params)
best_model.fit(train, show_progress=True)

# Сохранение в формате, который ожидает потом main.py
np.save(os.path.join(SRC_PATH, "als_user_factors.npy"), best_model.user_factors)
np.save(os.path.join(SRC_PATH, "als_item_factors.npy"), best_model.item_factors)

with open(os.path.join(SRC_PATH, "id_mappings.pkl"), 'wb') as f:
    pickle.dump({
        'user_id_map': user_id_map,
        'book_id_map': book_id_map,
        'user_id_inv': user_id_inv,
        'book_id_inv': book_id_inv
    }, f)

print(f"Все модели сохранены в: {ARTIFACTS_PATH}")
print(f"Лучшая модель и маппинги сохранены в src/model {SRC_PATH} ")

Разделение данных (80/20)...
Всего конфигураций для обучения: 1

[Model 1/1] Обучение с параметрами: {'factors': 32, 'regularization': 0.01, 'iterations': 30, 'alpha': 1.0}


100%|██████████| 30/30 [00:41<00:00,  1.39s/it]



СВОДНАЯ ТАБЛИЦА МЕТРИК
 model_id  factors  regularization  iterations  alpha  Precision@K   NDCG@K
        1       32            0.01          30    1.0     0.282557 0.326003

ЛУЧШАЯ МОДЕЛЬ (по NDCG@10): Model 1.0
factors           32.000000
regularization     0.010000
iterations        30.000000
alpha              1.000000
Precision@K        0.282557
NDCG@K             0.326003

Переобучение лучшей модели на full train и сохранение артефактов...


100%|██████████| 30/30 [00:44<00:00,  1.50s/it]


Все модели сохранены в: ../artifacts/als_models
Лучшая модель и маппинги сохранены в src/model ../src/models 
